# 05 Baselines: BKT, DKT and GKT on the same answers (C1 Adaptive Curriculum Engine)

The deep knowledge-tracing models the GAT is compared against, trained and scored on **exactly the same answers** as BKT.

| Model | Idea | Reference |
|---|---|---|
| BKT | Hidden learned / not learned state per exercise (parameters from notebook 03) | Corbett and Anderson 1994 |
| DKT | One LSTM over the whole answer history, predicts every exercise at the next step | Piech et al. 2015 |
| GKT | One hidden state per exercise, an answer updates that exercise **and its graph neighbours** (prerequisite graph from notebook 02) | Nakagawa et al. 2019 |

**Why PyTorch instead of the pyKT package.** pyKT expects its own preprocessing pipeline and file layout, and on Colab's Python 3.13 the BKT library (pyBKT) already failed to build its compiled part. DKT and GKT are written here in plain PyTorch following the papers (and pyKT's reference designs), so every model reads the same splits from notebook 01 and nothing depends on an unmaintained package.

**Protocol.** Histories are cut into chunks of 200 answers (notebook 01's `seq_chunk`), as in pyKT. Training students: folds 1 to 4; fold 0 picks the best epoch; scored once on the held-out test students. Every model is scored on the same answers: positions 2 to 200 of each test chunk (the first answer of a chunk has no history for DKT or GKT). A student-level bootstrap gives a 95% interval for each model's AUC difference from BKT.

Needs notebooks 01, 02 and 03. Use a **GPU runtime** (T4).

In [ ]:
# Where outputs are stored: "drive", "runtime" or "local" (see notebook 01).
STORAGE = "drive"
# True: 3,000 training and 1,000 test students, 1 epoch, for a fast check.
QUICK = False

In [ ]:
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn networkx

from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if not ON_COLAB:
    STORAGE = "local"

if STORAGE == "drive":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/adaptlearn-c1")
    except Exception as exc:
        print(f"Drive could not be mounted ({type(exc).__name__}: {exc}). Using runtime storage.")
        STORAGE = "runtime"
if STORAGE == "runtime":
    BASE = Path("/content/adaptlearn-c1")
elif STORAGE == "local":
    BASE = Path("../data").resolve()

PROCESSED = BASE / "junyi_processed"
OUT_DIR = PROCESSED / "baselines"
OUT_DIR.mkdir(parents=True, exist_ok=True)
for needed in ("interactions.parquet", "bkt/params.csv", "graph/edge_index.npy"):
    assert (PROCESSED / needed).exists(), f"{needed} missing: run notebooks 01, 02 and 03 first."
print("Storage:", STORAGE, "->", BASE)

## 1. Load answers, splits, the graph and the BKT parameters

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import torch

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CONFIG = {"SEED": SEED, "QUICK": QUICK, "EPOCHS": 1 if QUICK else 3,
          "DKT": {"EMB": 64, "HIDDEN": 128, "DROPOUT": 0.2, "BATCH": 128, "LR": 1e-3},
          "GKT": {"EMB": 32, "HIDDEN": 32, "DROPOUT": 0.2, "BATCH": 64, "LR": 1e-3},
          "BOOTSTRAP": 100}

inter = pd.read_parquet(PROCESSED / "interactions.parquet",
                        columns=["user_id", "exercise_idx", "step", "seq_chunk", "correct"])
inter = inter.sort_values(["user_id", "step"], kind="mergesort").reset_index(drop=True)
splits = json.loads((PROCESSED / "splits.json").read_text())
exercise_index = json.loads((PROCESSED / "exercise_index.json").read_text())
names = sorted(exercise_index, key=exercise_index.get)
N_EX = len(names)
edge_index = np.load(PROCESSED / "graph" / "edge_index.npy")  # row 0 prerequisite, row 1 exercise
bkt = pd.read_csv(PROCESSED / "bkt" / "params.csv").set_index("exercise").loc[names]

fold_of = {u: int(k) for k, users in splits["folds"].items() for u in users}
test_ids = set(splits["test"])
if QUICK:
    keep = (set(rng.choice(sorted(fold_of), min(3_000, len(fold_of)), replace=False))
            | set(rng.choice(sorted(test_ids), min(1_000, len(test_ids)), replace=False)))
    inter = inter[inter.user_id.isin(keep)].reset_index(drop=True)
print(f"{inter.user_id.nunique():,} students, {len(inter):,} answers, {N_EX} exercises, "
      f"{edge_index.shape[1]} prerequisite edges")

## 2. Chunks of up to 200 answers, and BKT's prediction for every answer

BKT runs over each student's whole history (it does not need chunks); its prediction for answer *t* uses answers before *t* only, with notebook 03's parameters.

In [ ]:
key = inter["user_id"].astype(str) + "|" + inter["seq_chunk"].astype(str)
chunk_id, _ = pd.factorize(key, sort=False)  # rows of a chunk are contiguous (sorted by student, time)
pos = inter.groupby(chunk_id, sort=False).cumcount().to_numpy()
n_chunks, L = int(chunk_id.max()) + 1, int(pos.max()) + 1
LEN = np.bincount(chunk_id)
Q = np.zeros((n_chunks, L), np.int64)
R = np.zeros((n_chunks, L), np.int64)
Q[chunk_id, pos] = inter["exercise_idx"].to_numpy()
R[chunk_id, pos] = inter["correct"].to_numpy()
first_row = np.r_[0, np.flatnonzero(np.diff(chunk_id)) + 1]
chunk_user = inter["user_id"].to_numpy()[first_row]
chunk_fold = np.array([fold_of.get(u, -1) for u in chunk_user])  # -1 = test student
is_test_chunk = np.isin(chunk_user, list(test_ids))
assert not (is_test_chunk & (chunk_fold >= 0)).any()
train_chunks = np.flatnonzero(chunk_fold >= 1)
val_chunks = np.flatnonzero(chunk_fold == 0)
test_chunks = np.flatnonzero(is_test_chunk)
print(f"{n_chunks:,} chunks of up to {L} answers: {len(train_chunks):,} train, "
      f"{len(val_chunks):,} validation, {len(test_chunks):,} test")

# BKT forward pass over each student's full history (same formula as the service).
prior, learn = bkt["p_init"].to_numpy(), bkt["p_learn"].to_numpy()
guess, slip = bkt["p_guess"].to_numpy(), bkt["p_slip"].to_numpy()
ex, y_all = inter["exercise_idx"].to_numpy(), inter["correct"].to_numpy().astype(bool)
users = inter["user_id"].to_numpy()
p_bkt = np.empty(len(inter))
start = time.time()
for i in range(len(inter)):
    if i == 0 or users[i] != users[i - 1]:
        state = prior.copy()
    e = ex[i]
    m = state[e]
    p_bkt[i] = m * (1 - slip[e]) + (1 - m) * guess[e]
    if y_all[i]:
        post = m * (1 - slip[e]) / (m * (1 - slip[e]) + (1 - m) * guess[e])
    else:
        post = m * slip[e] / (m * slip[e] + (1 - m) * (1 - guess[e]))
    state[e] = post + (1 - post) * learn[e]
P_BKT = np.zeros((n_chunks, L), np.float32)
P_BKT[chunk_id, pos] = p_bkt
print(f"BKT forward pass over {len(inter):,} answers in {time.time() - start:.0f} s")

## 3. The models

**DKT**: each answer (exercise, right or wrong) is embedded, an LSTM reads the history, and a linear layer gives a logit for every exercise; the logit of the exercise answered next is the prediction.

**GKT**: every exercise has its own hidden state. An answer to exercise *c* builds a message for *c* itself and for each graph neighbour (its prerequisites and its dependents, with separate networks per direction), passes it through an erase-add gate, and updates only those states with a GRU. The prediction for the next answer is read from the hidden state of the exercise answered next.

In [ ]:
import torch.nn as nn
import torch.nn.functional as fn


class DKT(nn.Module):
    def __init__(self, n, emb, hidden, dropout):
        super().__init__()
        self.emb = nn.Embedding(2 * n, emb)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, n)

    def forward(self, q, r):
        """q, r [B, T] -> logits [B, T-1] for answers 1..T-1."""
        h, _ = self.lstm(self.emb(q * 2 + r))
        logits = self.out(self.drop(h[:, :-1]))
        return logits.gather(-1, q[:, 1:].unsqueeze(-1)).squeeze(-1)


def neighbour_table(n, edges):
    """Padded neighbours per exercise; kind 1 = dependent (c -> j), 2 = prerequisite (j -> c), 0 = padding.
    Padding points at an extra dummy node n that is never read for a prediction."""
    nbrs = [[] for _ in range(n)]
    for p, c in edges.T.tolist():
        nbrs[p].append((c, 1))
        nbrs[c].append((p, 2))
    D = max(1, max(len(x) for x in nbrs))
    idx, kind = np.full((n, D), n, np.int64), np.zeros((n, D), np.int64)
    for v, lst in enumerate(nbrs):
        for k, (u, t) in enumerate(lst):
            idx[v, k], kind[v, k] = u, t
    return torch.from_numpy(idx), torch.from_numpy(kind)


def mlp(d_in, d_out, dropout):
    return nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_out, d_out))


class GKT(nn.Module):
    def __init__(self, n, edges, emb, hidden, dropout):
        super().__init__()
        self.n, self.H = n, hidden
        idx, kind = neighbour_table(n, edges)
        self.register_buffer("nb_idx", idx)
        self.register_buffer("nb_kind", kind)
        self.x_emb = nn.Embedding(2 * n, emb)
        self.c_emb = nn.Embedding(n + 1, emb)  # + the dummy padding node
        self.h0 = nn.Parameter(torch.zeros(n + 1, hidden))
        self.f_self = mlp(hidden + emb, hidden, dropout)
        self.f_out = mlp(2 * (hidden + emb), hidden, dropout)  # c -> j: c is a prerequisite of j
        self.f_in = mlp(2 * (hidden + emb), hidden, dropout)   # j -> c: j is a prerequisite of c
        self.erase = nn.Linear(hidden, hidden)
        self.add = nn.Linear(hidden, hidden)
        self.gru = nn.GRUCell(hidden, hidden)
        self.pred = nn.Linear(hidden + emb, 1)

    def forward(self, q, r):
        B, T = q.shape
        H, D = self.H, self.nb_idx.shape[1]
        rows = torch.arange(B, device=q.device)
        h = self.h0.unsqueeze(0).expand(B, -1, -1).contiguous()  # [B, n+1, H], row n = dummy
        logits = []
        for t in range(T - 1):
            c = q[:, t]
            x = self.x_emb(c * 2 + r[:, t])
            hc = h[rows, c]
            self_in = torch.cat([hc, x], -1)                                  # [B, H+E]
            kind = self.nb_kind[c]                                             # [B, D]
            idx = self.nb_idx[c]                                               # padding = dummy node n
            hj = h.gather(1, idx.unsqueeze(-1).expand(-1, -1, H))              # [B, D, H]
            nb_in = torch.cat([self_in.unsqueeze(1).expand(-1, D, -1), hj, self.c_emb(idx)], -1)
            m_nb = torch.where((kind == 1).unsqueeze(-1), self.f_out(nb_in), self.f_in(nb_in))
            m = torch.cat([self.f_self(self_in).unsqueeze(1), m_nb], 1)        # [B, 1+D, H]
            m = m * (1 - torch.sigmoid(self.erase(m))) + torch.tanh(self.add(m))
            old = torch.cat([hc.unsqueeze(1), hj], 1)
            new = self.gru(m.reshape(-1, H), old.reshape(-1, H)).view(B, 1 + D, H)
            # Real exercises appear once per step; only the dummy node repeats, and it is never read.
            h = h.scatter(1, torch.cat([c.unsqueeze(1), idx], 1).unsqueeze(-1).expand(-1, -1, H), new)
            nxt = q[:, t + 1]
            logits.append(self.pred(torch.cat([h[rows, nxt], self.c_emb(nxt)], -1)).squeeze(-1))
        return torch.stack(logits, 1)


print("DKT parameters:", sum(p.numel() for p in DKT(N_EX, 64, 128, 0.2).parameters()))
print("GKT parameters:", sum(p.numel() for p in GKT(N_EX, edge_index, 32, 32, 0.2).parameters()),
      "| max neighbours per exercise:", neighbour_table(N_EX, edge_index)[0].shape[1])

In [ ]:
from sklearn.metrics import accuracy_score, mean_squared_error, roc_auc_score

T_Q, T_R, T_LEN = torch.from_numpy(Q), torch.from_numpy(R), torch.from_numpy(LEN)


def batches(chunks, size, shuffle, gen=None):
    order = chunks[torch.randperm(len(chunks), generator=gen).numpy()] if shuffle else chunks
    for b in range(0, len(order), size):
        sel = order[b:b + size]
        T = int(LEN[sel].max())
        q, r = T_Q[sel, :T].to(DEVICE), T_R[sel, :T].to(DEVICE)
        mask = (torch.arange(1, T)[None, :] < T_LEN[sel][:, None]).to(DEVICE)  # answers 1..len-1
        yield sel, q, r, mask


@torch.no_grad()
def predict(model, chunks, size):
    """Probabilities for answers 1..len-1 of each chunk, in a fixed order, with labels and BKT."""
    model.eval()
    probs, labels, bkt_p, owner = [], [], [], []
    for sel, q, r, mask in batches(chunks, size, shuffle=False):
        p = torch.sigmoid(model(q, r))[mask].cpu().numpy()
        probs.append(p)
        labels.append(r[:, 1:][mask].cpu().numpy())
        T = q.shape[1]
        m = mask.cpu().numpy()
        bkt_p.append(P_BKT[sel, 1:T][m])
        owner.append(np.repeat(chunk_user[sel], m.sum(1)))
    return np.concatenate(probs), np.concatenate(labels), np.concatenate(bkt_p), np.concatenate(owner)


def fit(name, model, cfg):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["LR"])
    gen = torch.Generator().manual_seed(SEED)
    best_auc, best_state = -1.0, None
    for epoch in range(CONFIG["EPOCHS"]):
        model.train()
        start, total, count = time.time(), 0.0, 0
        for _, q, r, mask in batches(train_chunks, cfg["BATCH"], shuffle=True, gen=gen):
            logits = model(q, r)
            loss = fn.binary_cross_entropy_with_logits(logits[mask], r[:, 1:][mask].float())
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total += loss.item() * int(mask.sum())
            count += int(mask.sum())
        p, y, _, _ = predict(model, val_chunks, cfg["BATCH"] * 2)
        val_auc = roc_auc_score(y, p)
        print(f"{name} epoch {epoch + 1}: train loss {total / count:.4f}, validation AUC {val_auc:.4f}, "
              f"{(time.time() - start) / 60:.1f} min")
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_auc

## 4. Train DKT and GKT, score everything on the same test answers

In [ ]:
torch.manual_seed(SEED)
dkt, dkt_val = fit("DKT", DKT(N_EX, **{k.lower(): v for k, v in CONFIG["DKT"].items()
                                     if k in ("EMB", "HIDDEN", "DROPOUT")}), CONFIG["DKT"])
p_dkt, y_test, p_bkt_test, test_owner = predict(dkt, test_chunks, 256)

torch.manual_seed(SEED)
gkt, gkt_val = fit("GKT", GKT(N_EX, edge_index, **{k.lower(): v for k, v in CONFIG["GKT"].items()
                                                 if k in ("EMB", "HIDDEN", "DROPOUT")}), CONFIG["GKT"])
p_gkt, y_gkt, _, _ = predict(gkt, test_chunks, 128)
assert np.array_equal(y_gkt, y_test), "models were scored on different answers"

In [ ]:
def row(name, p):
    return {"model": name, "auc": roc_auc_score(y_test, p), "accuracy": accuracy_score(y_test, p >= 0.5),
            "rmse": float(np.sqrt(mean_squared_error(y_test, p)))}

majority = float(R[train_chunks].sum() / LEN[train_chunks].sum())  # training correct rate
results = pd.DataFrame([
    {"model": "majority class", "auc": 0.5, "accuracy": accuracy_score(y_test, np.full(len(y_test), majority >= 0.5)),
     "rmse": float(np.sqrt(mean_squared_error(y_test, np.full(len(y_test), majority))))},
    row("BKT (notebook 03)", p_bkt_test), row("DKT", p_dkt), row("GKT", p_gkt),
])

# 95% interval for each model's AUC minus BKT's, resampling test students (not answers).
students = np.unique(test_owner)
rows_of = pd.Series(np.arange(len(test_owner))).groupby(test_owner).apply(np.asarray).to_dict()
boot_rng = np.random.default_rng(SEED)
diffs = {"DKT": [], "GKT": []}
for _ in range(CONFIG["BOOTSTRAP"]):
    take = np.concatenate([rows_of[s] for s in boot_rng.choice(students, len(students), replace=True)])
    base = roc_auc_score(y_test[take], p_bkt_test[take])
    diffs["DKT"].append(roc_auc_score(y_test[take], p_dkt[take]) - base)
    diffs["GKT"].append(roc_auc_score(y_test[take], p_gkt[take]) - base)
intervals = {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in diffs.items()}
results["auc_minus_bkt_95ci"] = [None, None, str(np.round(intervals["DKT"], 4)), str(np.round(intervals["GKT"], 4))]
print(f"{len(y_test):,} test answers from {len(students):,} students (answers 2 to 200 of each chunk)")
results.round(4)

## 5. Save results and a manifest

In [ ]:
import hashlib, platform, sklearn
from datetime import datetime, timezone

results.to_csv(OUT_DIR / "test_results.csv", index=False)
(OUT_DIR / "summary.json").write_text(json.dumps({
    "test_answers": int(len(y_test)), "test_students": int(len(students)),
    "validation_auc": {"DKT": float(dkt_val), "GKT": float(gkt_val)},
    "auc_minus_bkt_95ci": intervals,
    "scored_rows": "answers 2 to 200 of each 200-answer chunk of the held-out test students",
}, indent=2))
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "scikit-learn": sklearn.__version__}
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG,
    "versions": VERSIONS,
    "sha256": {f.name: hashlib.sha256(f.read_bytes()).hexdigest()
               for f in sorted(OUT_DIR.glob("*")) if f.is_file() and f.name != "manifest.json"},
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

In [ ]:
if STORAGE == "runtime":
    import shutil
    print("Download before stopping the runtime:", shutil.make_archive("/content/baseline_outputs", "zip", OUT_DIR))
else:
    print("Outputs stored in", OUT_DIR)

## For the report

Report the results table (AUC primary; accuracy against the majority class) and the 95% intervals of the AUC difference from BKT. DKT and GKT are scored on answers 2 to 200 of each chunk, so their BKT row here differs slightly from notebook 03's (which scores every answer). The GAT is compared on the same footing in the results notebook.

Method sentence: *DKT (Piech et al., 2015) and GKT (Nakagawa et al., 2019) were implemented in PyTorch, trained on folds 1 to 4 with fold 0 for epoch selection, and scored on the held-out test students; GKT used the Junyi prerequisite graph from notebook 02.*

Before committing: **Clear all outputs**.